# QC Downstream Prediction

This notebook shows how to run NIQE-style image quality prediction and artifact prediction after extracting Eva patch features.

## Weights and Heads

Expected files:

```text
weights/NIQE.pth
weights/artifact.pth
```

Class definitions:

- `NIQE.pth`: `0=High_Quality`, `1=Low_Quality`
- `artifact.pth`: `0=No_Artifact`, `1=Artifact`

In [ ]:
# change work dir
import os

os.chdir("/autofs/bal14/yfliu/projects/Eva")

In [ ]:
from pathlib import Path

import torch
from omegaconf import OmegaConf

from Eva.utils import load_from_hf, extract_features

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
def load_qc_head(weight_path: str | Path, device: str | torch.device) -> torch.nn.Module:
    head = torch.nn.Sequential(
        torch.nn.LayerNorm(768, eps=1e-6),
        torch.nn.Linear(768, 2),
    )
    checkpoint = torch.load(weight_path, map_location="cpu", weights_only=False)
    head.load_state_dict(checkpoint["model_state_dict"])
    head.to(device)
    head.eval()
    return head

niqe_head = load_qc_head("weights/NIQE.pth", device)
artifact_head = load_qc_head("weights/artifact.pth", device)

## Extract Eva Features

If you already have extracted features, skip this section. The QC heads expect a tensor of shape `[num_patches, 768]`.

Use `cls=False` for both QC heads. If your input patch is already MIF-only, keep `channel_mode="full"` and do not remove the last three channels again.

In [ ]:
conf = OmegaConf.load("config.yaml")
model = load_from_hf(repo_id="yandrewl/Eva", conf=conf, device=device)
model.eval()

# Shape: [batch, height, width, channels]
patch = torch.randn(1, 224, 224, 6)
biomarkers = [["DAPI", "CD3e", "CD20", "CD4", "CD8", "PanCK"]]

with torch.no_grad():
    features = extract_features(
        patch=patch,
        bms=biomarkers,
        model=model,
        device=device,
        cls=False,
        channel_mode="full",
    )

features.shape

In [ ]:
with torch.no_grad():
    niqe_logits = niqe_head(features)
    artifact_logits = artifact_head(features)

    niqe_probs = torch.softmax(niqe_logits, dim=1)
    artifact_probs = torch.softmax(artifact_logits, dim=1) 

    niqe_labels = torch.argmax(niqe_probs, dim=1)
    artifact_labels = torch.argmax(artifact_probs, dim=1)

print("NIQE low-quality probability:", niqe_probs[:, 1])
print("NIQE label, 0=High_Quality, 1=Low_Quality:", niqe_labels)

print("Artifact probability:", artifact_probs[:, 1])
print("Artifact label, 0=No_Artifact, 1=Artifact:", artifact_labels)